# Notebook 11: Syntax Trees, Speaker Attribution, Lexicon, and Role Search

This notebook covers six major features added since Notebook 10:

| # | Feature | Module |
|---|---|---|
| 1 | [NT Greek Syntax (MACULA)](#1-nt-greek-syntax-macula) | `syntax` |
| 2 | [OT Hebrew Syntax (MACULA)](#2-ot-hebrew-syntax-macula) | `syntax_ot` |
| 3 | [Speaker Attribution](#3-speaker-attribution) | `speaker` |
| 4 | [Lexicon API](#4-lexicon-api) | `lexicon` |
| 5 | [Christological Titles](#5-christological-titles) | `christological_titles` |
| 6 | [Syntactic Role Search](#6-syntactic-role-search) | `role_search` |

**Data sources:**
- MACULA Greek Nestle1904 — 137,779 word tokens, syntactic roles, participant referents, glosses
- MACULA Hebrew WLC — 475,911 word tokens, clause types, inline LXX alignment per word

Both datasets are cached as Parquet files after the first load (~5–10 seconds each).

**Export as shareable HTML:**
```bash
jupyter nbconvert --to html notebooks/11_syntax_and_roles.ipynb
```

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print('Ready.')

---
## 1. NT Greek Syntax (MACULA)

The `syntax` module wraps the MACULA Greek Nestle1904 syntax tree data.
Each word token carries:

- **`role`** — syntactic role: `s` (subject), `v` (verb), `o` (object), `io` (indirect object), `p` (predicate), `vc` (verb complement), `adv` (adverbial)
- **`subjref`** — xml_id of the grammatical subject of this verb
- **`referent`** — xml_id of the referent this word points to (for pronouns, articles, etc.)
- **`gloss`** — English gloss
- **`domain`** — Louw-Nida semantic domain
- **`frame`** — semantic frame label
- **`tense`, `voice`, `mood`, `case_`, `person`, `number`, `gender`** — full morphology

The first call to `load_syntax()` parses the TSV and writes a Parquet cache.

In [ ]:
from bible_grammar import load_syntax, query_syntax

df_nt = load_syntax()
print(f'NT syntax tokens: {len(df_nt):,}')
print(f'Columns: {list(df_nt.columns)}')

In [ ]:
# John 1:1 — "In the beginning was the Word"
# Columns of interest: text, lemma, role, gloss, tense, case_
jhn1 = query_syntax(book='Jhn', chapter=1, verse=1)
jhn1[['ref', 'text', 'lemma', 'strong_g', 'role', 'gloss', 'case_', 'tense']]

In [ ]:
# Distribution of syntactic roles across the NT
df_nt['role'].value_counts()

In [ ]:
# How many verb tokens have a resolved grammatical subject?
verbs = df_nt[df_nt['class_'] == 'verb']
has_subj = verbs['subjref'].notna().sum()
print(f'Total NT verb tokens:          {len(verbs):>7,}')
print(f'With resolved subject (subjref): {has_subj:>7,}  ({100*has_subj/len(verbs):.1f}%)')

In [ ]:
from bible_grammar import speech_verbs, jesus_speaking_verses

# Find speech verbs (λέγω, φημί, λαλέω, ἀποκρίνομαι…) in Matthew
sv = speech_verbs('Mat', subject_strong='2424')   # G2424 = Iesous
print(f'Speech verbs with Jesus as subject in Matthew: {len(sv)}')
sv[['ref', 'text', 'lemma', 'gloss']].head(10)

In [ ]:
# Full set of verses where Jesus is the grammatical subject of a speech verb
gospel_books = ['Mat', 'Mrk', 'Luk', 'Jhn']
speaking = jesus_speaking_verses(gospel_books)
print(f'Distinct (book, chapter, verse) tuples where Jesus speaks: {len(speaking)}')

---
## 2. OT Hebrew Syntax (MACULA)

The `syntax_ot` module wraps the MACULA Hebrew WLC data (475,911 tokens across
930 per-chapter lowfat XML files). Each word carries:

- **`type_`** — clause type: `wayyiqtol`, `qatal`, `yiqtol`, `imperative`, `participle`, `infinitive`, `nominal`, etc.
- **`stem`** — binyan: `qal`, `niphal`, `piel`, `pual`, `hiphil`, `hophal`, `hithpael`
- **`role`** — syntactic role (same codes as NT: s/v/o/…)
- **`subjref`** / **`participantref`** — xml_id links to grammatical subject/participant
- **`greek`** / **`greek_g`** — the LXX Greek word and Strong's number that translates *this specific token*
- **`lang`** — `H` (Hebrew) or `A` (Aramaic)

The inline LXX alignment is word-level from the syntax tree — higher precision than IBM Model 1.

In [ ]:
from bible_grammar import load_syntax_ot, query_syntax_ot

df_ot = load_syntax_ot()
print(f'OT syntax tokens: {len(df_ot):,}')
print(f'Columns: {list(df_ot.columns)}')

In [ ]:
# Genesis 1:1 — "In the beginning God created the heavens and the earth"
gen1 = query_syntax_ot(book='Gen', chapter=1, verse=1)
gen1[['ref', 'text', 'lemma', 'strong_h', 'role', 'stem', 'type_', 'greek', 'greek_g']]

In [ ]:
# Clause type distribution across the OT — the backbone of Hebrew grammar
df_ot[df_ot['pos'] == 'verb']['type_'].value_counts().head(15)

In [ ]:
# wayyiqtol (narrative waw-consecutive past) in Genesis
wayyiqtol_gen = query_syntax_ot(book='Gen', tense='wayyiqtol')
print(f'wayyiqtol verbs in Genesis: {len(wayyiqtol_gen):,}')
wayyiqtol_gen[['ref', 'text', 'lemma', 'gloss', 'stem']].head(10)

In [ ]:
# Aramaic sections: Daniel 2:4–7:28, Ezra 4:8–6:18, 7:12–26
aramaic = query_syntax_ot(lang='A')
print(f'Aramaic tokens: {len(aramaic):,}')
aramaic.groupby('book').size().sort_values(ascending=False)

In [ ]:
from bible_grammar import lxx_alignment

# How does the LXX render שָׁלוֹם (H7965, peace)?
# Uses inline tree alignment — word-level, not statistical
lxx_alignment('H7965')

In [ ]:
# רוּחַ (H7307, spirit/wind) — how does the LXX resolve the ambiguity?
lxx_alignment('H7307')

In [ ]:
# חֶסֶד (H2617, lovingkindness/steadfast love) — no single English word covers its range
# The LXX usually renders it ἔλεος (mercy)
lxx_alignment('H2617')

---
## 3. Speaker Attribution

The `speaker` module identifies which NT verses contain Jesus speaking,
using two complementary strategies:

1. **Curated allowlists** — frozensets of exact (book, chapter, verse) triples
   for well-known, invariant verse sets: Son of Man sayings, Johannine I AM
   declarations, bridegroom parables. These are 100%-confident.

2. **MACULA subjref detection** — speech verbs (λέγω, φημί, λαλέω, ἀποκρίνομαι,
   εἶπεν, ὁμολογέω, κράζω) whose grammatical subject resolves to Iesous (G2424)
   via the MACULA `subjref` syntax tree attribute.

The two strategies are complementary: allowlists are precise for specific titles;
MACULA detection gives broad coverage across the Gospels and Acts.

In [ ]:
from bible_grammar import jesus_speaking_verse_set, is_jesus_speaking, ALLOWLIST_VERSES

# All Gospel verses where Jesus speaks (MACULA detection)
speaking_set = jesus_speaking_verse_set(['Mat', 'Mrk', 'Luk', 'Jhn'])
print(f'Verses with Jesus as speaking subject (Gospels): {len(speaking_set)}')

In [ ]:
# Curated allowlists — titles with hand-curated verse sets
print('Allowlist titles:')
for title, verses in ALLOWLIST_VERSES.items():
    print(f'  {title}: {len(verses)} verses')

In [ ]:
# Check individual verses
test_verses = [
    ('Mat', 16, 13),   # "Who do people say the Son of Man is?" — Jesus speaking
    ('Jhn', 8, 58),    # "Before Abraham was born, I AM" — Johannine I AM
    ('Mat', 3, 17),    # "This is my Son" — God speaking, not Jesus
    ('Jhn', 11, 35),   # "Jesus wept" — narration, not speech
]

for book, ch, vs in test_verses:
    result = is_jesus_speaking(book, ch, vs)
    print(f'{book} {ch}:{vs:2d}  →  Jesus speaking: {result}')

---
## 4. Lexicon API

The `lexicon` module provides a clean public API for the TBESH (Hebrew) and TBESG (Greek)
Translators Brief lexicons included in the STEPBible data. It centralizes the parsing logic
that was previously duplicated across several modules.

Each lexicon entry contains: Strong's number, lemma, transliteration, part-of-speech code,
one-line gloss, and a full definition.

In [ ]:
from bible_grammar import lookup, lex_entry

# Lookup returns a dict
entry = lookup('H7965')   # שָׁלוֹם
print(entry)

In [ ]:
# Pretty-print a full lexicon article
lex_entry('H2617')   # חֶסֶד — lovingkindness / steadfast love

In [ ]:
lex_entry('G3056')   # λόγος — word / reason

In [ ]:
from bible_grammar import search_gloss

# Search the Hebrew lexicon by gloss keyword
results = search_gloss('covenant', lang='H')
print('Hebrew entries matching "covenant":')
for r in results:
    print(f"  {r['strongs']:>7}  {r['lemma']:<12}  {r['gloss']}")

In [ ]:
# Search the Greek lexicon by gloss keyword
results = search_gloss('faith', lang='G')
print('Greek entries matching "faith":')
for r in results:
    print(f"  {r['strongs']:>7}  {r['lemma']:<16}  {r['gloss']}")

In [ ]:
from bible_grammar import lemma_index

# Reverse lookup: lemma → Strong's number
heb_idx = lemma_index('H')
print(f'Hebrew lemma index: {len(heb_idx):,} entries')
print(f'שָׁלוֹם → {heb_idx.get("שָׁלוֹם")}')
print(f'בְּרִית → {heb_idx.get("בְּרִית")}')

grk_idx = lemma_index('G')
print(f'Greek lemma index: {len(grk_idx):,} entries')
print(f'λόγος → {grk_idx.get("λόγος")}')
print(f'ἀγάπη → {grk_idx.get("ἀγάπη")}')

---
## 5. Christological Titles

The `christological_titles` module counts how frequently Jesus used various titles
to refer to Himself across the four Gospels. It includes an optional **speaker filter**
that restricts counts to verses where Jesus is the grammatical subject of a speech
verb — ensuring we count only self-referential usage, not narrator or other characters'
references to Jesus.

**Tracked titles:**
- Son of Man (ὁ υἱὸς τοῦ ἀνθρώπου) — Jesus' most frequent self-designation
- I AM sayings (ἐγώ εἰμι with predicates, Johannine)
- Son of God (υἱὸς τοῦ θεοῦ)
- Lord / Kyrios (Κύριος used self-referentially)
- Bridegroom (νυμφίος)
- Teacher / Rabbi (διδάσκαλος, ῥαββί)
- Good Shepherd (ποιμήν)
- Light of the World (φῶς τοῦ κόσμου)

In [ ]:
from bible_grammar import print_title_counts, title_counts

# Unfiltered: all occurrences of each title pattern in the Gospels
print('=== All Gospel occurrences (unfiltered) ===')
print_title_counts(scope='gospels', speaker_filter=False)

In [ ]:
# Filtered: only where Jesus is the speaking subject (MACULA subjref + allowlists)
# This removes narrator references and other characters' use of the titles
print('=== Jesus speaking only (speaker_filter=True) ===')
print_title_counts(scope='gospels', speaker_filter=True)

In [ ]:
# As a DataFrame for further analysis
df_titles = title_counts(scope='gospels', speaker_filter=True)
df_titles

In [ ]:
from bible_grammar import title_chart
from IPython.display import Image

chart_path = title_chart(scope='gospels', speaker_filter=True,
                         output_path='../output/charts/christological-titles-filtered.png')
print(f'Chart saved: {chart_path}')
Image(chart_path)

In [ ]:
from bible_grammar import title_verses

# All I AM sayings in John (with verse references)
verses = title_verses('I AM')
print(f'Johannine I AM sayings: {len(verses)}')
for ref in sorted(verses)[:10]:
    print(f'  {ref[0]} {ref[1]}:{ref[2]}')

---
## 6. Syntactic Role Search

The `role_search` module answers "who does what to whom" by following MACULA
`subjref` links. Given one or more Strong's numbers, it finds all verb tokens
whose grammatical subject resolves to one of those entities.

This enables questions like:
- What verbs does YHWH/Elohim take as subject across the entire OT?
- What does Jesus do in the Gospels (syntactically)?
- Which verbs are *only* ever attributed to divine subjects?
- How does divine action language in the OT compare to the NT?

In [ ]:
from bible_grammar import subject_verbs, print_role_summary

# What does God (YHWH + Elohim) do in the OT?
print_role_summary(['H3068', 'H0430'], corpus='OT', top_n=20, label='YHWH+Elohim')

In [ ]:
# As a DataFrame — includes lemma, gloss, stem, LXX Greek equivalent
df_god_ot = subject_verbs(['H3068', 'H0430'], corpus='OT', top_n=30)
df_god_ot[['lemma', 'gloss', 'stem', 'greek', 'greek_g', 'count']].head(20)

In [ ]:
# What does God (Theos) do in the NT?
print_role_summary(['G2316'], corpus='NT', top_n=20, label='Theos')

In [ ]:
# What does Jesus (Iesous) do in the Gospels?
print_role_summary(['G2424'], corpus='NT', books=['Mat', 'Mrk', 'Luk', 'Jhn'],
                   top_n=20, label='Iesous')

In [ ]:
# YHWH's verbs in Isaiah only
print_role_summary(['H3068'], corpus='OT', books=['Isa'],
                   top_n=15, label='YHWH (Isaiah)')

In [ ]:
from bible_grammar import verb_subjects

# Who does בָּרָא (bara, to create) take as its grammatical subject?
# Classical question: does any human ever take bara as subject?
bara_subjects = verb_subjects('H1254', corpus='OT')
print('Grammatical subjects of בָּרָא (create) in the OT:')
bara_subjects

In [ ]:
from bible_grammar import role_chart
from IPython.display import Image

chart_path = role_chart(['H3068', 'H0430'], corpus='OT', top_n=20,
                        label='YHWH+Elohim',
                        output_path='../output/charts/role-yhwh-elohim-ot.png')
print(f'Chart saved: {chart_path}')
Image(chart_path)

In [ ]:
from bible_grammar import divine_action_comparison
from IPython.display import Image

# Side-by-side: God's verbs in OT Hebrew vs NT Greek
# OT column shows Hebrew lemmas + inline LXX Greek equivalents
ot_df, nt_df, chart_path = divine_action_comparison(
    output_path='../output/charts/divine-action-ot-nt.png'
)
print(f'Chart saved: {chart_path}')
Image(chart_path)

In [ ]:
from bible_grammar import role_report

# Generate full Markdown report: top verbs, book distribution, OT/NT comparison
report_path = role_report(
    ['H3068', 'H0430'], corpus='OT', top_n=30,
    label='YHWH+Elohim',
    output_dir='../output/reports',
    include_cross_testament=True
)
print(f'Report: {report_path}')

---
## Quick Reference — Features Added in Notebook 11

```python
from bible_grammar import (
    # NT Syntax (MACULA Greek)
    load_syntax,               # 137k-row DataFrame, Parquet-cached
    query_syntax,              # filter by book/chapter/verse/strong/role/tense/…
    speech_verbs,              # speech-verb tokens with given subject
    jesus_speaking_verses,     # set of (book, ch, vs) where Jesus speaks
    MACULA_BOOK_MAP,           # MAT→Mat, MRK→Mrk, … mapping dict

    # OT Syntax (MACULA Hebrew)
    load_syntax_ot,            # 475k-row DataFrame, Parquet-cached
    query_syntax_ot,           # filter by book/chapter/verse/strong_h/stem/tense/lang/…
    lxx_alignment,             # LXX rendering frequency for any Hebrew Strong's number
    clause_roles_ot,           # syntactic roles for a single verse
    MACULA_OT_BOOK_MAP,        # GEN→Gen, EXO→Exo, … mapping dict

    # Speaker Attribution
    is_jesus_speaking,         # bool — is Jesus the speaking subject in this verse?
    jesus_speaking_verse_set,  # frozenset of (book, ch, vs) tuples
    filter_to_jesus_speech,    # filter a verse list to Jesus-only speech
    ALLOWLIST_VERSES,          # dict of curated verse sets by title

    # Lexicon API
    lookup,                    # dict entry for any Strong's number
    search_gloss,              # search lexicon by English gloss keyword
    lex_entry,                 # pretty-print a full lexicon article
    lemma_index,               # {lemma → strongs} reverse lookup dict

    # Christological Titles
    title_counts,              # DataFrame of title frequencies per Gospel book
    print_title_counts,        # formatted terminal report
    title_chart,               # bar chart PNG
    title_verses,              # set of (book, ch, vs) for a given title
    title_report,              # full Markdown report
    TITLE_REGISTRY,            # dict of title definitions and search patterns

    # Syntactic Role Search
    subject_verbs,             # verbs with given Strong's number(s) as grammatical subject
    verb_subjects,             # subjects that take a given verb
    print_role_summary,        # formatted terminal table
    role_chart,                # horizontal bar PNG
    divine_action_comparison,  # side-by-side OT/NT divine action chart
    role_report,               # full Markdown report
    GOD_OT,                    # {'H0430','H3068','H0136','H0410'}
    GOD_NT,                    # {'G2316'}
    JESUS_NT,                  # {'G2424'}
)
```

**Slash commands** (Claude Code): `/role-search`, `/christological-titles`

**Pre-generated reports** (`output/reports/`):
- `role-yhwh-elohim-ot.md` — YHWH+Elohim as OT subject, top 30 verbs with LXX equivalents
- `divine-action-ot-nt.png` — side-by-side OT/NT divine action chart
- `christological-titles.md` — title frequency across the four Gospels